###Read data from table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
df = spark.table("bronze_layer_data.erp_px_cat_g1v2")


## Silver Transformation 

Trimming

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)


Renaming Columns 

In [0]:

RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Sanity Checks 

In [0]:

df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver_layer_data.erp_product_category")

In [0]:

%sql
SELECT * FROM silver_layer_data.erp_product_category LIMIT 10;